In [1]:
from __future__ import annotations
import requests, certifi
from pathlib import Path
import csv, io, hashlib
from datetime import datetime
from pathlib import Path
from typing import Optional, Dict, List, Tuple
import re
import requests


# Bronze layer

In [2]:
# --- Incremental Bronze with ETag/Last-Modified state --------------------------------

# ====== Config ======
BASE_URL = "https://balanca.economia.gov.br/balanca/bd/comexstat-bd/ncm"
KINDS = ("EXP", "IMP")
START_YEAR = 1997
END_YEAR = 2025
TIMEOUT = 90
CHUNK = 1 << 15
VERIFY = True  # set False only for quick tests in SSL-inspected networks (not for prod)

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://balanca.economia.gov.br/",
}

# ROOT = Path("data")
# BRONZE = ROOT / "bronze"
# MANIFESTS = ROOT / "manifests"
# for p in (BRONZE, MANIFESTS):
#     p.mkdir(parents=True, exist_ok=True)

# ROOT = Path("data")
BRONZE = Path(r'C:\Users\matheus.manzke\Projetos\dados-comex\Dados\bronze')
MANIFESTS = Path(r'C:\Users\matheus.manzke\Projetos\dados-comex\Dados\manifests')
for p in (BRONZE, MANIFESTS):
    p.mkdir(parents=True, exist_ok=True)

STATE_PATH = MANIFESTS / "bronze_state.csv"     # one row per (kind,year)
MANIFEST_PATH = MANIFESTS / "bronze_manifest.csv"

# ====== Small helpers ======
def url_for(kind: str, year: int) -> str:
    return f"{BASE_URL}/{kind}_{year}.csv"

def bronze_path(kind: str, year: int) -> Path:
    return BRONZE / kind / f"{kind}_{year}.csv"

def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_csv_map(path: Path) -> Dict[Tuple[str,int], Dict[str,str]]:
    if not path.exists():
        return {}
    rows = {}
    with path.open("r", encoding="utf-8", newline="") as f:
        r = csv.DictReader(f)
        for row in r:
            rows[(row["kind"], int(row["year"]))] = row
    return rows

def append_csv_row(path: Path, row: Dict) -> None:
    write_header = not path.exists()
    with path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            w.writeheader()
        w.writerow(row)

def try_head(url: str) -> Optional[requests.Response]:
    try:
        r = requests.head(url, headers=HEADERS, timeout=TIMEOUT, allow_redirects=True, verify=VERIFY)
        return r if r.status_code == 200 else None
    except requests.RequestException:
        return None

def discover_years(kind: str, start_year: int = START_YEAR, end_year: Optional[int] = None) -> List[int]:
    if end_year is None:
        end_year = datetime.utcnow().year
    found = []
    for y in range(start_year, end_year + 1):
        if try_head(url_for(kind, y)):
            found.append(y)
    return found

def extract_remote_meta(resp: requests.Response) -> Dict[str, str]:
    """Grab lightweight identity of the remote object from headers."""
    h = resp.headers
    return {
        "etag": h.get("ETag", ""),
        "last_modified": h.get("Last-Modified", ""),
        "content_length": h.get("Content-Length", ""),
    }

# ====== Decision policy ======================================================
def needs_download(prev: Optional[Dict[str,str]], remote: Dict[str,str]) -> bool:
    """
    Decide if we should download:
    - If no previous state → download
    - If ETag changed → download
    - Else if Last-Modified changed → download
    - Else if Content-Length changed → download
    - Else → skip
    """
    if prev is None:
        return True
    # Compare in order of reliability
    if remote.get("etag") and remote["etag"] != prev.get("etag"):
        return True
    if remote.get("last_modified") and remote["last_modified"] != prev.get("last_modified"):
        return True
    if remote.get("content_length") and remote["content_length"] != prev.get("content_length"):
        return True
    return False

# ====== Bronze core with incremental logic ==================================
def download_csv(kind: str, year: int) -> Path:
    """Stream download to temp, atomic rename on success."""
    out_path = bronze_path(kind, year)
    ensure_parent(out_path)
    url = url_for(kind, year)
    with requests.get(url, headers=HEADERS, stream=True, timeout=TIMEOUT, verify=VERIFY) as r:
        r.raise_for_status()
        tmp = out_path.with_suffix(".downloading")
        with tmp.open("wb") as f:
            for chunk in r.iter_content(chunk_size=CHUNK):
                if chunk:
                    f.write(chunk)
        tmp.replace(out_path)
    return out_path

def validate_csv_basic(path: Path) -> Dict:
    size = path.stat().st_size
    if size == 0:
        raise ValueError(f"Empty file: {path}")
    with path.open("rb") as f:
        head = f.read(512 * 1024)
    sample = io.TextIOWrapper(io.BytesIO(head), encoding="utf-8", errors="replace")
    reader = csv.reader(sample)
    try:
        header = next(reader)
    except StopIteration:
        raise ValueError("CSV has no header")
    data_rows = 0
    for _ in range(10):
        try:
            row = next(reader)
        except StopIteration:
            break
        if row:
            data_rows += 1
    if data_rows == 0:
        raise ValueError("No data rows found in the first lines")
    return {
        "size_bytes": size,
        "n_cols": len(header),
        "header_sample": "|".join(header[:20]),
        "n_rows_sampled": data_rows,
    }

def bronze_ingest_incremental(kind: str, year: int) -> Path:
    """
    Incremental ingest for one (kind,year):
      - HEAD → remote meta
      - Compare to previous state (bronze_state.csv)
      - If unchanged → SKIP (record manifest with action=skip_unchanged)
      - If changed/new → download, validate, hash, update state, write manifest
    """
    url = url_for(kind, year)
    ts = datetime.utcnow().isoformat(timespec="seconds") + "Z"

    # Discover remote meta (HEAD)
    head_resp = try_head(url)
    if head_resp is None:
        raise RuntimeError(f"Remote not available (no 200 on HEAD): {url}")
    remote_meta = extract_remote_meta(head_resp)

    # Load prior state row (if any)
    state_map = read_csv_map(STATE_PATH)
    prev = state_map.get((kind, year))

    action = "skip_unchanged"
    path = bronze_path(kind, year)

    if needs_download(prev, remote_meta) or not path.exists():
        # Download with up to 3 tries
        last_err = None
        for attempt in range(3):
            try:
                path = download_csv(kind, year)
                meta = validate_csv_basic(path)
                file_hash = sha256_file(path)
                action = "downloaded_new" if prev is None else "replaced_changed"
                # Update state row
                state_row = {
                    "kind": kind,
                    "year": year,
                    "etag": remote_meta.get("etag", ""),
                    "last_modified": remote_meta.get("last_modified", ""),
                    "content_length": remote_meta.get("content_length", ""),
                    "sha256": file_hash,
                    "path": str(path),
                    "ts_updated": ts,
                }
                append_csv_row(STATE_PATH, state_row)
                # Audit manifest
                manifest_row = {
                    "ts": ts,
                    "stage": "bronze",
                    "action": action,
                    "kind": kind,
                    "year": year,
                    "url": url,
                    "path": str(path),
                    "sha256": file_hash,
                    **meta,
                }
                append_csv_row(MANIFEST_PATH, manifest_row)
                print(f"✔ {action}: {kind}_{year} → {path}")
                return path
            except Exception as e:
                last_err = e
                print(f"Attempt {attempt+1}/3 failed for {kind}_{year}: {e}")
        raise RuntimeError(f"Failed bronze ingest for {kind}_{year}: {last_err}")
    else:
        # Unchanged → record a lightweight manifest entry
        manifest_row = {
            "ts": ts,
            "stage": "bronze",
            "action": action,
            "kind": kind,
            "year": year,
            "url": url,
            "path": str(path),
            "sha256": prev.get("sha256", "") if prev else "",
            "size_bytes": path.stat().st_size if path.exists() else "",
            "n_cols": "",
            "header_sample": "",
            "n_rows_sampled": "",
        }
        append_csv_row(MANIFEST_PATH, manifest_row)
        print(f"↩ skipped (unchanged): {kind}_{year}")
        return path



In [3]:
# ====== Example runs =========================================================
# 1) One-off backfill for all existing years (use sparingly; first run only)
def bronze_backfill_all():
    for kind in KINDS:
        years = discover_years(kind)
        for y in years:
            bronze_ingest_incremental(kind, y)

# 2) Monthly incremental (typical scheduler target)
def bronze_monthly_incremental():
    this_year = datetime.utcnow().year
    for kind in KINDS:
        if try_head(url_for(kind, this_year)):
            bronze_ingest_incremental(kind, this_year)



In [5]:
bronze_backfill_all()

C:\Users\matheus.manzke\AppData\Local\Temp\ipykernel_27780\1288672244.py:76: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_year = datetime.utcnow().year
C:\Users\matheus.manzke\AppData\Local\Temp\ipykernel_27780\1288672244.py:167: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat(timespec="seconds") + "Z"


↩ skipped (unchanged): EXP_1997
↩ skipped (unchanged): EXP_1998
↩ skipped (unchanged): EXP_1999
↩ skipped (unchanged): EXP_2000
↩ skipped (unchanged): EXP_2001
↩ skipped (unchanged): EXP_2002
↩ skipped (unchanged): EXP_2003
↩ skipped (unchanged): EXP_2004
↩ skipped (unchanged): EXP_2005
↩ skipped (unchanged): EXP_2006
↩ skipped (unchanged): EXP_2007
↩ skipped (unchanged): EXP_2008
↩ skipped (unchanged): EXP_2009
↩ skipped (unchanged): EXP_2010
↩ skipped (unchanged): EXP_2011
↩ skipped (unchanged): EXP_2012
↩ skipped (unchanged): EXP_2013
↩ skipped (unchanged): EXP_2014
↩ skipped (unchanged): EXP_2015
↩ skipped (unchanged): EXP_2016
↩ skipped (unchanged): EXP_2017
↩ skipped (unchanged): EXP_2018
↩ skipped (unchanged): EXP_2019
↩ skipped (unchanged): EXP_2020
↩ skipped (unchanged): EXP_2021
↩ skipped (unchanged): EXP_2022
↩ skipped (unchanged): EXP_2023
↩ skipped (unchanged): EXP_2024
✔ replaced_changed: EXP_2025 → C:\Users\matheus.manzke\Projetos\dados-comex\Dados\bronze\EXP\EXP_2025.cs

In [4]:
bronze_monthly_incremental()

C:\Users\matheus.manzke\AppData\Local\Temp\ipykernel_27928\1279630038.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  this_year = datetime.utcnow().year
C:\Users\matheus.manzke\AppData\Local\Temp\ipykernel_27928\1288672244.py:167: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat(timespec="seconds") + "Z"


✔ replaced_changed: EXP_2026 → C:\Users\matheus.manzke\Projetos\dados-comex\Dados\bronze\EXP\EXP_2026.csv
✔ replaced_changed: IMP_2026 → C:\Users\matheus.manzke\Projetos\dados-comex\Dados\bronze\IMP\IMP_2026.csv


Silver layer

In [5]:
from pathlib import Path
import duckdb, re, os


QT_STAT!!!!!!!

In [6]:
# Robust silver step for EXP/IMP with schema drift across years
# - Peeks header to see which columns exist in THIS file
# - Builds a safe SELECT (missing columns become NULL)
# - Dedups on the subset of key columns that actually exist
# - Writes Parquet without loading whole CSV into Python RAM


BRONZE = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\bronze")
SILVER  = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver")
SILVER.mkdir(parents=True, exist_ok=True)

def _pick(cols, *cands):
    """First candidate present in cols, else None."""
    for c in cands:
        if c in cols:
            return c
    return None

def _q(c):
    """Quote identifier if present, else None."""
    return f'"{c}"' if c else None

def _num_int(c):    return f"TRY_CAST(NULLIF(TRIM({c}), '') AS INTEGER)" if c else "NULL"
def _num_bigint(c): return f"TRY_CAST(NULLIF(TRIM({c}), '') AS BIGINT)"  if c else "NULL"
def _num_double(c): return f"TRY_CAST(NULLIF(TRIM({c}), '') AS DOUBLE)"  if c else "NULL"
def _txt(c):        return f"NULLIF(TRIM({c}), '')" if c else "NULL"

def silver_one(kind: str, year: int):
    src = (BRONZE / kind / f"{kind}_{year}.csv").resolve()
    dst = (SILVER  / kind / f"{kind}_{year}.parquet").resolve()
    dst.parent.mkdir(parents=True, exist_ok=True)

    if not src.exists():
        print(f"skip: {src} not found")
        return

    con = duckdb.connect()
    # con.execute("PRAGMA memory_limit='1GB'; PRAGMA threads=4;")

    # 1) Peek header only to know actual column names in THIS file
    header_df = con.execute(f"""
        SELECT *
        FROM read_csv_auto('{src.as_posix()}',
                           HEADER=TRUE,
                           IGNORE_ERRORS=TRUE,
                           SAMPLE_SIZE=1,      -- header only
                           ALL_VARCHAR=TRUE)
        LIMIT 1
    """).fetchdf()
    cols = set(header_df.columns)

    # 2) Map common core columns (use upper/lower variants if present)
    CO_ANO   = _pick(cols, "CO_ANO","co_ano")
    CO_MES   = _pick(cols, "CO_MES","co_mes")
    CO_NCM   = _pick(cols, "CO_NCM","co_ncm")
    CO_PAIS  = _pick(cols, "CO_PAIS","co_pais")
    CO_UNID  = _pick(cols, "CO_UNID","co_unid")
    KG_LIQ   = _pick(cols, "KG_LIQUIDO","kg_liquido","KG_LIQ","kg_liq")
    VL_FOB   = _pick(cols, "VL_FOB","vl_fob","VL_VALOR","vl_valor")  # some years use different labels
    QT_ESTAT   = _pick(cols, "QT_ESTAT","qt_estat")  # some years use different labels


    # 3) Useful descriptors if present (safe optional)
    SG_UF    = _pick(cols, "SG_UF_NCM","sg_uf_ncm","SG_UF","sg_uf")

    # 4) Build dedup PARTITION BY list using only columns that exist
    key_exprs = []
    if CO_ANO:  key_exprs.append(_num_int(_q(CO_ANO)))
    if CO_MES:  key_exprs.append(_num_int(_q(CO_MES)))
    if CO_NCM:  key_exprs.append(_num_bigint(_q(CO_NCM)))
    if CO_PAIS: key_exprs.append(_num_int(_q(CO_PAIS)))
    if KG_LIQ:  key_exprs.append(_num_double(_q(KG_LIQ)))
    if VL_FOB:  key_exprs.append(_num_double(_q(VL_FOB)))
    if QT_ESTAT:  key_exprs.append(_num_double(_q(QT_ESTAT)))


    # Always include injected partition context so duplicates across kind/year don’t collide
    key_exprs += [f"'{kind}'", f"{year}"]

    partition_by = ",\n              ".join(key_exprs)

    # 5) Assemble SELECT with safe casts (missing -> NULL)
    sql = f"""
    COPY (
      SELECT
        '{kind}'::VARCHAR AS kind,
        {year}::INTEGER   AS year,

        {_num_int(   _q(CO_ANO))}   AS co_ano,
        {_num_int(   _q(CO_MES))}   AS co_mes,
        {_num_bigint(_q(CO_NCM))} AS co_ncm,
        CASE
        WHEN {_num_bigint(_q(CO_NCM))} IS NULL THEN NULL
        ELSE LPAD(CAST({_num_bigint(_q(CO_NCM))} AS VARCHAR), 8, '0')
        END AS ncm8,
        {_num_int(   _q(CO_PAIS))}  AS co_pais,
        {_num_int(   _q(CO_UNID))}  AS co_unid,
        {_num_double(_q(KG_LIQ))}   AS kg_liquido,
        {_num_double(_q(VL_FOB))}   AS vl_fob,
        {_num_double(_q(QT_ESTAT))}   AS qt_estat,

        {_txt(       _q(SG_UF))}    AS sg_uf_ncm,

        ROW_NUMBER() OVER (
                        PARTITION BY
                            {partition_by}
                        ORDER BY
                            (vl_fob IS NULL), (kg_liquido IS NULL),   -- prefer non-null rows
                            vl_fob DESC, kg_liquido DESC              -- then highest values
                        ) AS _rn

      FROM read_csv_auto('{src.as_posix()}',
                         HEADER=TRUE,
                         IGNORE_ERRORS=TRUE,
                         SAMPLE_SIZE=-1,
                         ALL_VARCHAR=TRUE)
      QUALIFY _rn = 1
    )
    TO '{dst.as_posix()}'
    (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
    """

    con.execute(sql)
    con.close()
    print(f"silver ✔ {dst}")


In [7]:
BRONZE = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\bronze")
fname_re = re.compile(r"^(EXP|IMP)_(\d{4})\.csv$", re.IGNORECASE)

todo = []
for kind_dir in (BRONZE/"EXP", BRONZE/"IMP"):
    if kind_dir.exists():
        for p in sorted(kind_dir.glob("*.csv")):
            m = fname_re.match(p.name)
            if m:
                todo.append((m.group(1).upper(), int(m.group(2))))

for kind, year in todo:
    silver_one(kind, year)

silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_1997.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_1998.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_1999.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_2000.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_2001.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_2002.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_2003.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_2004.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_2005.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_2006.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_2007.parquet
silver ✔ C:\Users\matheus.manzke

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_2025.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\EXP\EXP_2026.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_1997.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_1998.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_1999.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_2000.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_2001.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_2002.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_2003.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_2004.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_2005.parquet
silver ✔ C:\Users\matheus.manzke

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_2011.parquet


...

silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_2023.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_2024.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_2025.parquet
silver ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\IMP\IMP_2026.parquet


Gold merge

In [8]:
from pathlib import Path
import pandas as pd
import duckdb


In [9]:
pd.read_csv(r'C:\Users\matheus.manzke\Projetos\dados_comex\data\silver\DICT\PAIS.csv', sep=";", encoding='latin1')

,CO_PAIS,CO_PAIS_ISON3,CO_PAIS_ISOA3,NO_PAIS,NO_PAIS_ING,NO_PAIS_ESP
0,0,898,ZZZ,Não Definido,Not defined,No definido
1,13,4,AFG,Afeganistão,Afghanistan,Afganistan
2,15,248,ALA,"Aland, Ilhas",Aland Islands,"Alans, Islas"
3,17,8,ALB,Albânia,Albania,Albania
4,20,724,ESP,"Alboran-Perejil, Ilhas","Alboran-Perejil, Islands","Alboran-Perejil, Islas"
...,...,...,...,...,...,...
276,994,898,ZZZ,A Designar,To define,A designar
277,995,898,ZZZ,Bancos Centrais,Central Banks,Bancos Centrales
278,997,898,ZZZ,Organizações Internacionais,International Organizations,Organizaciones Internacionales
279,998,898,ZZZ,Sem informação,Sem informação,Sem informação


In [10]:
DICT_NCM_XLSX = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver\DICT\NCM_SH4 - atualizado 07.11.25.xlsx")
DICT_PAIS_XLSX = Path(r"C:\Users\matheus.manzke\Projetos\dados_comex\data\silver\DICT\PAIS.csv")
GOLD_DIR = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\expimp").resolve()
SILVER = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\silver").resolve()

def build_gold_partitioned_and_merged_sc():
    """Stack all silver Parquets → join NCM dict → partitioned gold + one merged table."""
    con = duckdb.connect()
    # con.execute("PRAGMA memory_limit='1GB'; PRAGMA threads=4;")
    GOLD_DIR.mkdir(parents=True, exist_ok=True)

    # --- 1) Load dictionary from Excel (robust via pandas) and register in DuckDB
    # Keep only columns we need; normalize key to 8-char string
    dict_ncm_cols = ["Código NCM 8 dígitos", "Produto", "SC Competitiva", "CNAE divisão"]
    dim_ncm_df = pd.read_excel(DICT_NCM_XLSX, usecols=dict_ncm_cols, dtype={"Código NCM 8 dígitos": "string"})
    # normalize/clean the NCM key (Excel may store as number; strip .0, pad left)
    dim_ncm_df["ncm8"] = (
        dim_ncm_df["Código NCM 8 dígitos"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
        .str.zfill(8)
    )
    # Drop empties/dupes on the key (keep first)
    dim_ncm_df = dim_ncm_df.dropna(subset=["ncm8"]).drop_duplicates(subset=["ncm8"], keep="first")
    dim_ncm_df = dim_ncm_df.rename(columns={
        "Produto": "produto",
        "SC Competitiva": "sc_competitiva",
        "CNAE divisão": "cnae_divisao"
    })

    dict_pais_cols = ['CO_PAIS', 'CO_PAIS_ISON3', 'CO_PAIS_ISOA3', 'NO_PAIS']
    dim_pais_df = pd.read_csv(r'C:\Users\matheus.manzke\Projetos\dados_comex\data\silver\DICT\PAIS.csv', sep=";", encoding='latin1', usecols=dict_pais_cols)

    # Normalize columns
    def _to_int_or_none(x: str):
      if x is None: return None
      s = str(x).strip()
      if s.endswith(".0"): s = s[:-2]
      return pd.to_numeric(s, errors="coerce")

    # Drop empties/dupes on the key (keep first)
    dim_pais_df = dim_pais_df.rename(columns={
    "NO_PAIS"       : "nm_pais",        # ← you were missing a comma here
    "CO_PAIS"       : "co_pais",
    "CO_PAIS_ISON3" : "co_pais_ison3",
    "CO_PAIS_ISOA3" : "co_pais_isoa3",
})

    # Clean values
    dim_pais_df["co_pais"]        = dim_pais_df["co_pais"].map(_to_int_or_none).astype("Int64")
    dim_pais_df["nm_pais"]        = dim_pais_df["nm_pais"].str.strip()
    # dim_pais_df["co_pais_ison3"]  = (
    #     dim_pais_df["co_pais_ison3"].str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(3)
    # )  # keep ISO numeric as 3-char string (e.g., 076)
    dim_pais_df["co_pais_isoa3"]  = dim_pais_df["co_pais_isoa3"].str.strip().str.upper()


    dim_pais_df = dim_pais_df.dropna(subset=["co_pais"]).drop_duplicates(subset=["co_pais"], keep="first")

    # Register dim tables 
    con.register("dim_ncm", dim_ncm_df)  # columns: ncm8, produto, sc_competitiva, cnae_divisao, Código NCM 8 dígitos
    con.register("dim_pais", dim_pais_df) 

    # --- 2) Build the fact+dict joined SELECT once (used for both writes)
    silver_glob = (SILVER / "*/*.parquet").as_posix()

    joined_select = f"""
  WITH fact AS (
  SELECT
    *,
    CASE
      WHEN co_ncm IS NULL THEN NULL
      ELSE LPAD(CAST(co_ncm AS VARCHAR), 8, '0')
    END AS ncm8,

  FROM read_parquet('{silver_glob}', union_by_name=true)
  WHERE (vl_fob IS NULL OR vl_fob >= 0)
    AND (kg_liquido IS NULL OR kg_liquido >= 0)
)
SELECT
  f.*,
  d.produto,
  d.sc_competitiva,
  d.cnae_divisao,
  COALESCE(
    NULLIF(TRIM(CAST(p.nm_pais AS VARCHAR)), ''),

  ) AS nm_pais_dim,
  p.co_pais_ison3,
  p.co_pais_isoa3
FROM fact f
LEFT JOIN dim_ncm  AS d ON f.ncm8 = d.ncm8
LEFT JOIN dim_pais AS p ON CAST(f.co_pais AS INTEGER) = p.co_pais
"""

    # --- 3) Partitioned write (Hive-style): kind=…/year=…/part-*.parquet
    con.execute(f"""
      COPY (
        {joined_select}
      )
      TO '{GOLD_DIR.as_posix()}'
      (FORMAT PARQUET, PARTITION_BY (kind, year), OVERWRITE_OR_IGNORE 1);
    """)

    # --- 4) Single merged file (all years & kinds)
    merged_path = GOLD_DIR.parent / "comexstat_ncm_all.parquet"
    con.execute(f"""
      COPY (
        SELECT * FROM read_parquet('{GOLD_DIR.as_posix()}/kind=*/year=*/*.parquet', union_by_name=true)
      )
      TO '{merged_path.as_posix()}'
      (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
    """)

    # --- 5) Quick summary
    rows_total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{merged_path.as_posix()}')").fetchone()[0]
    sample = con.execute(f"""
        SELECT kind, year, COUNT(*) AS n
        FROM read_parquet('{merged_path.as_posix()}')
        GROUP BY 1,2
        ORDER BY year DESC, kind
        LIMIT 20
    """).fetchdf()
    con.close()

    print(f"gold ✔ partitioned at: {GOLD_DIR}")
    print(f"gold ✔ merged file   : {merged_path}  (rows={rows_total:,})")
    display(sample)


In [11]:
# 2) Build gold (partitioned + merged single file)
build_gold_partitioned_and_merged_sc()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

gold ✔ partitioned at: C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\expimp
gold ✔ merged file   : C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\comexstat_ncm_all.parquet  (rows=76,311,932)


,kind,year,n
0,EXP,2026,247159
1,IMP,2026,379393
2,EXP,2025,1705924
3,IMP,2025,2393547
4,EXP,2024,1600823
5,IMP,2024,2269822
6,EXP,2023,1560952
7,IMP,2023,2085295
8,EXP,2022,1493675
9,IMP,2022,2020286


In [12]:
import polars as pl

Criando tabela SC

In [13]:
ncm_all = pl.read_parquet(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\comexstat_ncm_all.parquet")

In [14]:
ncm_all = ncm_all.filter(pl.col('sg_uf_ncm')=='SC')

In [15]:
ncm_all.write_parquet(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\comexstat_ncm_sc.parquet")

# Data Marts

In [16]:
from pathlib import Path
import duckdb

TOP PRODUTOS - ORDEM CORRIGIDA

In [17]:
GOLD_DIR  = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\expimp").resolve()
MARTS_DIR = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\marts\jan").resolve()
MARTS_DIR.mkdir(parents=True, exist_ok=True)

def build_mart_exp_top_products_report(
    ref_year: int | None = None,
    ref_month: int | None = None,
    kind: str = "EXP",
    out_name: str = "mart_exp_top_products_report.parquet",
):
    flow_kind = (kind or "EXP").upper()
    if flow_kind not in ("EXP", "IMP"):
        raise ValueError("kind must be 'EXP' or 'IMP'")

    con = duckdb.connect()
    gold_glob = rf"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\comexstat_ncm_sc.parquet"

    # Detect latest month if not provided
    if ref_year is None or ref_month is None:
        d_curr = con.execute(f"""
            SELECT MAX(MAKE_DATE(CAST(year AS INTEGER), CAST(co_mes AS INTEGER), 1))
            FROM read_parquet('{gold_glob}', union_by_name=true)
        """).fetchone()[0]
        if d_curr is None:
            con.close()
            raise RuntimeError("No data found in Gold to determine reference month.")
        ref_year, ref_month = d_curr.year, d_curr.month

    # Portuguese month label helpers
    meses_pt = ["Janeiro","Fevereiro","Março","Abril","Maio","Junho",
                "Julho","Agosto","Setembro","Outubro","Novembro","Dezembro"]
    mes_nome = meses_pt[ref_month - 1]
    prev_label_val = f"{mes_nome} {ref_year-1} (em US$ FOB)"
    curr_label_val = f"{mes_nome} {ref_year} (em US$ FOB)"
    curr_label_kg  = f"{mes_nome} {ref_year} (em quilogramas líquidos)"

    def ident(s: str) -> str:
        return '"' + s.replace('"', '""') + '"'

    final_sql = f"""
    WITH base AS (
      SELECT
        kind,
        CAST(year AS INTEGER)     AS yr,
        CAST(co_mes AS INTEGER)   AS mo,
        co_ncm,
        ncm8,
        co_pais,
        COALESCE(CAST(nm_pais_dim AS VARCHAR)) AS pais_nome,
        produto,
        UPPER(TRIM(produto))      AS produto_norm,
        TRIM(produto)             AS produto_clean,
        sc_competitiva,
        cnae_divisao,
        CAST(vl_fob     AS DECIMAL(20,4)) AS vl_fob,
        CAST(kg_liquido AS DECIMAL(20,6)) AS kg_liquido,
        CAST(qt_estat   AS DECIMAL(20,6)) AS qt_estat,
        MAKE_DATE(CAST(year AS INTEGER), CAST(co_mes AS INTEGER), 1) AS d
      FROM read_parquet('{gold_glob}', union_by_name=true)
      WHERE (vl_fob IS NULL OR vl_fob >= 0)
        AND (kg_liquido IS NULL OR kg_liquido >= 0)
    ),
    month_scope AS (
      SELECT
        MAKE_DATE({int(ref_year)}, {int(ref_month)}, 1) AS d_curr,
        MAKE_DATE({int(ref_year)}, {int(ref_month)}, 1) - INTERVAL 1 YEAR AS d_prev
    ),
    -- Rank Top-20 products by CURRENT MONTH value only
    prod_rank_curr AS (
      SELECT
        UPPER(TRIM(b.produto)) AS produto_norm,
        MIN(TRIM(b.produto))   AS produto_display,
        SUM(b.vl_fob)          AS vl_fob_curr_month
      FROM base b
      JOIN month_scope m ON b.d = m.d_curr
      WHERE b.kind = '{flow_kind}' 
        AND b.produto IS NOT NULL 
        AND TRIM(b.produto) <> ''
      GROUP BY 1
    ),
    top20_prod AS (
      SELECT *
      FROM (
        SELECT *,
               ROW_NUMBER() OVER (ORDER BY vl_fob_curr_month DESC, produto_norm) AS rn
        FROM prod_rank_curr
      ) WHERE rn <= 20
    ),
    -- Main destination per product (for the CURRENT MONTH)
    main_dest AS (
      SELECT
        UPPER(TRIM(b.produto)) AS produto_norm,
        b.co_pais,
        MIN(b.pais_nome)       AS pais_nome,
        SUM(b.vl_fob)          AS vl_fob_dest_curr,
        ROW_NUMBER() OVER (
          PARTITION BY UPPER(TRIM(b.produto))
          ORDER BY SUM(b.vl_fob) DESC, b.co_pais
        ) AS rnk
      FROM base b
      JOIN month_scope m ON b.d = m.d_curr
      JOIN top20_prod t ON UPPER(TRIM(b.produto)) = t.produto_norm
      WHERE b.kind = '{flow_kind}'
      GROUP BY UPPER(TRIM(b.produto)), b.co_pais
    ),
    main_dest_pick AS (
      SELECT
        produto_norm,
        pais_nome AS principal_destino
      FROM main_dest
      WHERE rnk = 1
    ),
    -- Current month metrics
    curr_m AS (
      SELECT
        UPPER(TRIM(b.produto)) AS produto_norm,
        SUM(b.vl_fob)          AS vl_fob_curr,
        SUM(b.kg_liquido)      AS kg_curr,
        SUM(b.qt_estat)        AS qt_curr
      FROM base b
      JOIN month_scope m ON b.d = m.d_curr
      WHERE b.kind = '{flow_kind}' 
        AND b.produto IS NOT NULL 
        AND TRIM(b.produto) <> ''
      GROUP BY 1
    ),
    -- Previous year same month metrics
    prev_m AS (
      SELECT
        UPPER(TRIM(b.produto)) AS produto_norm,
        SUM(b.vl_fob)          AS vl_fob_prev,
        SUM(b.kg_liquido)      AS kg_prev,
        SUM(b.qt_estat)        AS qt_prev
      FROM base b
      JOIN month_scope m ON b.d = m.d_prev
      WHERE b.kind = '{flow_kind}' 
        AND b.produto IS NOT NULL 
        AND TRIM(b.produto) <> ''
      GROUP BY 1
    )
    SELECT
      t.produto_display                                        AS {ident('Produtos Valor FOB US$')},
      COALESCE( CAST(p.vl_fob_prev AS DECIMAL(20,2)), 0 )      AS {ident(prev_label_val)},
      COALESCE( CAST(c.vl_fob_curr AS DECIMAL(20,2)), 0 )      AS {ident(curr_label_val)},
      COALESCE( CAST(c.kg_curr     AS DECIMAL(20,3)), 0 )      AS {ident(curr_label_kg)},
      CASE
        WHEN p.vl_fob_prev IS NULL OR p.vl_fob_prev = 0 THEN NULL
        ELSE (c.vl_fob_curr - p.vl_fob_prev) / p.vl_fob_prev
      END                                                     AS {ident('Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %)')},
      CASE
        WHEN p.kg_prev IS NULL OR p.kg_prev = 0 THEN NULL
        ELSE (c.kg_curr - p.kg_prev) / p.kg_prev
      END                                                     AS {ident('Variação de Quantidade Exportada - Quilograma Líquido (em %)')},
      CASE
        WHEN (p.kg_prev IS NULL OR p.kg_prev = 0) THEN NULL
        WHEN (c.kg_curr IS NULL OR c.kg_curr = 0) THEN NULL
        ELSE
          ((c.vl_fob_curr / NULLIF(c.kg_curr, 0)) - (p.vl_fob_prev / NULLIF(p.kg_prev, 0)))
          / NULLIF((p.vl_fob_prev / NULLIF(p.kg_prev, 0)), 0)
      END                                                     AS {ident('Variação do Preço Médio')},
      CASE
        WHEN p.qt_prev IS NULL OR p.qt_prev = 0 THEN NULL
        ELSE (c.qt_curr - p.qt_prev) / p.qt_prev
      END                                                     AS {ident('Variação da Quantidade Estatística')},
      md.principal_destino                                    AS {ident('Principal destino dos produtos mais exportados por SC')}
    FROM top20_prod t
    LEFT JOIN main_dest_pick md ON md.produto_norm = t.produto_norm
    LEFT JOIN curr_m c          ON c.produto_norm  = t.produto_norm
    LEFT JOIN prev_m p          ON p.produto_norm  = t.produto_norm
    ORDER BY t.rn
    """

    out_path = MARTS_DIR / out_name
    con.execute(f"""
      COPY ({final_sql})
      TO '{out_path.as_posix()}'
      (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1)
    """)

    # quick peek
    preview = con.execute(f"SELECT * FROM read_parquet('{out_path.as_posix()}') LIMIT 10").fetchdf()
    con.close()
    print(f"mart ✔ {out_path}")
    display(preview)


In [19]:
build_mart_exp_top_products_report(2026, 2, kind="EXP", out_name="mart_exp_top_products_report.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mart ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\marts\jan\mart_exp_top_products_report.parquet


,Produtos Valor FOB US$,Fevereiro 2025 (em US$ FOB),Fevereiro 2026 (em US$ FOB),Fevereiro 2026 (em quilogramas líquidos),Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %),Variação de Quantidade Exportada - Quilograma Líquido (em %),Variação do Preço Médio,Variação da Quantidade Estatística,Principal destino dos produtos mais exportados por SC
0,Carnes de aves,200208493.0,209524619.0,101873127.0,0.046532,-0.006786,0.053682,-0.007037,Países Baixos (Holanda)
1,Carne suína,140241109.0,128169533.0,49976772.0,-0.086077,-0.110986,0.028019,-0.110026,Japão
2,Motores elétricos,44244870.0,46692832.0,10311650.0,0.055328,0.033727,0.020895,0.149784,Estados Unidos
3,Tabaco não manufaturado,8429690.0,35973586.0,6038470.0,3.267486,4.390528,-0.208336,4.390528,China
4,Partes de motor,34275695.0,26328495.0,9556859.0,-0.231861,-0.333669,0.152789,0.219491,México
5,Madeira serrada,36075100.0,23768002.0,49235383.0,-0.341152,-0.332296,-0.013263,-0.258917,Estados Unidos
6,"Papel Kraft, não revestidos",15973179.0,20282691.0,31987303.0,0.269797,0.328836,-0.044429,0.328836,Argentina
7,Outras preparações e conservas de carnes e miu...,18109029.0,17939184.0,5289900.0,-0.009379,-0.091569,0.090474,-0.091569,Reino Unido
8,Transformadores elétricos,15934928.0,16804561.0,1122288.0,0.054574,-0.026394,0.083163,0.807601,Estados Unidos
9,Madeira compensada,18622450.0,15518156.0,28599413.0,-0.166696,-0.154806,-0.014068,-0.162344,Alemanha


In [21]:
build_mart_exp_top_products_report(2026, 2, kind="IMP", out_name="mart_imp_top_products_report.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mart ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\marts\jan\mart_imp_top_products_report.parquet


,Produtos Valor FOB US$,Fevereiro 2025 (em US$ FOB),Fevereiro 2026 (em US$ FOB),Fevereiro 2026 (em quilogramas líquidos),Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %),Variação de Quantidade Exportada - Quilograma Líquido (em %),Variação do Preço Médio,Variação da Quantidade Estatística,Principal destino dos produtos mais exportados por SC
0,Cobre refinado,137607960.0,109933164.0,8468007.0,-0.201113,-0.434600,0.412959,-0.441678,Chile
1,Pneus de borracha,41133085.0,77423157.0,32401280.0,0.882260,0.990478,-0.054368,0.413403,China
2,Partes e acessórios para veículos,60923591.0,76430814.0,8804823.0,0.254536,0.173296,0.069241,-0.162763,China
3,Revestimento de ferros laminados planos,34876110.0,49197631.0,72954718.0,0.410640,0.549751,-0.089763,0.549751,China
4,Polímeros de etileno,55224050.0,49172014.0,45838563.0,-0.109591,-0.039258,-0.073206,-0.039258,Argentina
5,Alumínio em formas brutas,27312284.0,47856997.0,15387248.0,0.752215,0.591436,0.101028,0.131400,Argentina
6,Fios de filamentos sintéticos,46227602.0,41859096.0,31611243.0,-0.094500,0.072924,-0.156045,0.072924,China
7,Fertilizantes nitrogenados,18580873.0,36128372.0,96458438.0,0.944385,0.697061,0.145736,0.697061,Omã
8,Semicondutores,37553712.0,32328783.0,19092011.0,-0.139132,-0.256101,0.157237,0.131745,China
9,Veículos,7527896.0,30439544.0,1529991.0,3.043566,2.618326,0.117524,1.600928,Estados Unidos


TOP DESTINOS ORDEM CORRIGIDA

In [22]:
from pathlib import Path
import duckdb

GOLD_DIR  = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\expimp").resolve()
MARTS_DIR = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\marts\jan").resolve()
MARTS_DIR.mkdir(parents=True, exist_ok=True)

def build_mart_exp_top_destinations_report(
    ref_year: int | None = None,
    ref_month: int | None = None,
    kind: str = "EXP",
    out_name: str = "mart_exp_top_destinations_report.parquet",
):
    """
    Produces a Top-20 destinations report (for EXP/IMP; default EXP) with columns:
      1) 'Destinos Valor FOB US$'                                  -- destination name
      2) '<Mês YYYY-1> (em US$ FOB)'                               -- prev year same month value
      3) '<Mês YYYY> (em US$ FOB)'                                 -- current month value
      4) '<Mês YYYY> (em quilogramas líquidos)'                    -- current month kg
      5) 'Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %)'
      6) 'Variação de Quantidade Exportada - Quilograma Líquido (em %)'
      7) 'Variação do Preço Médio'
      8) 'Variação da Quantidade Estatística'
      9) 'Produto mais exportado para o destino'                   -- top product for current month
    Ranking basis: sum(vl_fob) for the reference month.
    """

    flow_kind = (kind or "EXP").upper()
    if flow_kind not in ("EXP", "IMP"):
        raise ValueError("kind must be 'EXP' or 'IMP'")

    con = duckdb.connect()
    gold_glob = rf"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\comexstat_ncm_sc.parquet"

    # Detect latest month if not provided
    if ref_year is None or ref_month is None:
        d_curr = con.execute(f"""
            SELECT MAX(MAKE_DATE(CAST(year AS INTEGER), CAST(co_mes AS INTEGER), 1))
            FROM read_parquet('{gold_glob}', union_by_name=true)
        """).fetchone()[0]
        if d_curr is None:
            con.close()
            raise RuntimeError("No data found in Gold to determine reference month.")
        ref_year, ref_month = d_curr.year, d_curr.month

    # Month labels (pt-BR)
    meses_pt = ["Janeiro","Fevereiro","Março","Abril","Maio","Junho",
                "Julho","Agosto","Setembro","Outubro","Novembro","Dezembro"]
    mes_nome = meses_pt[ref_month - 1]
    prev_label_val = f"{mes_nome} {ref_year-1} (em US$ FOB)"
    curr_label_val = f"{mes_nome} {ref_year} (em US$ FOB)"
    curr_label_kg  = f"{mes_nome} {ref_year} (em quilogramas líquidos)"

    def ident(s: str) -> str:
        return '"' + s.replace('"', '""') + '"'

    final_sql = f"""
    WITH base AS (
      SELECT
        kind,
        CAST(year AS INTEGER)     AS yr,
        CAST(co_mes AS INTEGER)   AS mo,
        co_pais,
        COALESCE(CAST(nm_pais_dim AS VARCHAR)) AS pais_nome,
        produto,
        UPPER(TRIM(produto))      AS produto_norm,
        TRIM(produto)             AS produto_clean,
        CAST(vl_fob     AS DECIMAL(20,4)) AS vl_fob,
        CAST(kg_liquido AS DECIMAL(20,6)) AS kg_liquido,
        CAST(qt_estat   AS DECIMAL(20,6)) AS qt_estat,
        MAKE_DATE(CAST(year AS INTEGER), CAST(co_mes AS INTEGER), 1) AS d
      FROM read_parquet('{gold_glob}', union_by_name=true)
      WHERE (vl_fob IS NULL OR vl_fob >= 0)
        AND (kg_liquido IS NULL OR kg_liquido >= 0)
    ),
    month_scope AS (
      SELECT
        MAKE_DATE({int(ref_year)}, {int(ref_month)}, 1) AS d_curr,
        MAKE_DATE({int(ref_year)}, {int(ref_month)}, 1) - INTERVAL 1 YEAR AS d_prev
    ),

    -- Rank destinations by CURRENT MONTH value only
    dest_rank_curr AS (
      SELECT
        b.co_pais,
        MIN(b.pais_nome) AS pais_nome,
        SUM(b.vl_fob)    AS vl_fob_curr_month
      FROM base b
      JOIN month_scope m ON b.d = m.d_curr
      WHERE b.kind = '{flow_kind}'
      GROUP BY 1
    ),
    top20_dest AS (
      SELECT *
      FROM (
        SELECT *,
               ROW_NUMBER() OVER (ORDER BY vl_fob_curr_month DESC, co_pais) AS rn
        FROM dest_rank_curr
      ) WHERE rn <= 20
    ),

    -- Top product per destination for CURRENT MONTH
    main_prod AS (
      SELECT
        b.co_pais,
        UPPER(TRIM(b.produto)) AS produto_norm,
        MIN(TRIM(b.produto))   AS produto,
        SUM(b.vl_fob)          AS vl_fob_prod_curr,
        ROW_NUMBER() OVER (
          PARTITION BY b.co_pais
          ORDER BY SUM(b.vl_fob) DESC, UPPER(TRIM(b.produto))
        ) AS rnk
      FROM base b
      JOIN month_scope m ON b.d = m.d_curr
      JOIN top20_dest t ON b.co_pais = t.co_pais
      WHERE b.kind = '{flow_kind}'
        AND b.produto IS NOT NULL 
        AND TRIM(b.produto) <> ''
      GROUP BY b.co_pais, UPPER(TRIM(b.produto))
    ),
    main_prod_pick AS (
      SELECT
        co_pais,
        produto AS top_produto
      FROM main_prod
      WHERE rnk = 1
    ),

    -- Current month metrics
    curr_m AS (
      SELECT
        b.co_pais,
        SUM(b.vl_fob)     AS vl_fob_curr,
        SUM(b.kg_liquido) AS kg_curr,
        SUM(b.qt_estat)   AS qt_curr
      FROM base b
      JOIN month_scope m ON b.d = m.d_curr
      WHERE b.kind = '{flow_kind}'
      GROUP BY 1
    ),
    -- Previous year same month metrics
    prev_m AS (
      SELECT
        b.co_pais,
        SUM(b.vl_fob)     AS vl_fob_prev,
        SUM(b.kg_liquido) AS kg_prev,
        SUM(b.qt_estat)   AS qt_prev
      FROM base b
      JOIN month_scope m ON b.d = m.d_prev
      WHERE b.kind = '{flow_kind}'
      GROUP BY 1
    )

    -- Final: exactly the requested columns and order
    SELECT
      t.pais_nome                                            AS {ident('Destinos Valor FOB US$')},
      COALESCE( CAST(p.vl_fob_prev AS DECIMAL(20,2)), 0 )    AS {ident(prev_label_val)},
      COALESCE( CAST(c.vl_fob_curr AS DECIMAL(20,2)), 0 )    AS {ident(curr_label_val)},
      COALESCE( CAST(c.kg_curr     AS DECIMAL(20,3)), 0 )    AS {ident(curr_label_kg)},
      CASE
        WHEN p.vl_fob_prev IS NULL OR p.vl_fob_prev = 0 THEN NULL
        ELSE (c.vl_fob_curr - p.vl_fob_prev) / p.vl_fob_prev
      END                                                   AS {ident('Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %)')},
      CASE
        WHEN p.kg_prev IS NULL OR p.kg_prev = 0 THEN NULL
        ELSE (c.kg_curr - p.kg_prev) / p.kg_prev
      END                                                   AS {ident('Variação de Quantidade Exportada - Quilograma Líquido (em %)')},
      CASE
        WHEN (p.kg_prev IS NULL OR p.kg_prev = 0) THEN NULL
        WHEN (c.kg_curr IS NULL OR c.kg_curr = 0) THEN NULL
        ELSE
          ((c.vl_fob_curr / NULLIF(c.kg_curr, 0)) - (p.vl_fob_prev / NULLIF(p.kg_prev, 0)))
          / NULLIF((p.vl_fob_prev / NULLIF(p.kg_prev, 0)), 0)
      END                                                   AS {ident('Variação do Preço Médio')},
      CASE
        WHEN p.qt_prev IS NULL OR p.qt_prev = 0 THEN NULL
        ELSE (c.qt_curr - p.qt_prev) / p.qt_prev
      END                                                   AS {ident('Variação da Quantidade Estatística')},
      mp.top_produto                                        AS {ident('Produto mais exportado para o destino')}
    FROM top20_dest t
    LEFT JOIN main_prod_pick mp ON mp.co_pais = t.co_pais
    LEFT JOIN curr_m c          ON c.co_pais  = t.co_pais
    LEFT JOIN prev_m p          ON p.co_pais  = t.co_pais
    ORDER BY t.rn
    """

    out_path = MARTS_DIR / out_name
    con.execute(f"""
      COPY ({final_sql})
      TO '{out_path.as_posix()}'
      (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1)
    """)

    preview = con.execute(f"SELECT * FROM read_parquet('{out_path.as_posix()}') LIMIT 10").fetchdf()
    con.close()
    print(f"mart ✔ {out_path}")
    display(preview)

# Examples:
# build_mart_exp_top_destinations_report()        # EXP, latest month
# build_mart_exp_top_destinations_report(2025, 8) # EXP, labels "Agosto 2024/2025"
# build_mart_exp_top_destinations_report(kind="IMP")


In [24]:
build_mart_exp_top_destinations_report(2026, 2, kind="EXP", out_name="mart_exp_top_destinations_report.parquet")

mart ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\marts\jan\mart_exp_top_destinations_report.parquet


,Destinos Valor FOB US$,Fevereiro 2025 (em US$ FOB),Fevereiro 2026 (em US$ FOB),Fevereiro 2026 (em quilogramas líquidos),Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %),Variação de Quantidade Exportada - Quilograma Líquido (em %),Variação do Preço Médio,Variação da Quantidade Estatística,Produto mais exportado para o destino
0,Estados Unidos,128389959.0,84833412.0,39814272.0,-0.339252,-0.524188,0.388674,-0.385162,Motores elétricos
1,China,71678164.0,84458365.0,107021906.0,0.178300,0.211551,-0.027445,2.038914,Tabaco não manufaturado
2,Japão,55321977.0,64766014.0,22381614.0,0.170710,0.027241,0.139665,0.027309,Carne suína
3,Argentina,71338315.0,60443352.0,32235329.0,-0.152722,-0.195050,0.052584,-0.171718,"Papel Kraft, não revestidos"
4,México,43646814.0,50471386.0,32090340.0,0.156359,0.333397,-0.132772,0.213372,Carnes de aves
5,Países Baixos (Holanda),38311963.0,42557651.0,12056306.0,0.110819,-0.079378,0.206596,-0.103026,Carnes de aves
6,Chile,43365125.0,39946387.0,23074517.0,-0.078836,-0.099241,0.022653,-0.125356,Carne suína
7,Filipinas,37832808.0,37255530.0,23517167.0,-0.015259,-0.100501,0.094766,0.024906,Carne suína
8,Paraguai,34916147.0,30968703.0,17760232.0,-0.113055,-0.295439,0.258861,-0.326937,Cerâmica não vitrificada
9,Arábia Saudita,29089791.0,26137143.0,16458034.0,-0.101501,-0.305906,0.294491,0.094370,Carnes de aves


In [ ]:
build_mart_exp_top_destinations_report(2026, 2, kind="EXP", out_name="mart_exp_top_destinations_report.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mart ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\marts\jan\mart_exp_top_destinations_report.parquet


,Destinos Valor FOB US$,Janeiro 2025 (em US$ FOB),Janeiro 2026 (em US$ FOB),Janeiro 2026 (em quilogramas líquidos),Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %),Variação de Quantidade Exportada - Quilograma Líquido (em %),Variação do Preço Médio,Variação da Quantidade Estatística,Produto mais exportado para o destino
0,Japão,51585755.0,66686370.0,23146544.0,0.292728,0.129204,0.144813,0.130275,Carne suína
1,China,93213742.0,64995007.0,117238862.0,-0.302731,0.278697,-0.454704,1.586420,Carnes de aves
2,Estados Unidos,110158205.0,62741827.0,39327064.0,-0.430439,-0.489604,0.115919,-0.682639,Partes de motor
3,Argentina,71805272.0,47953642.0,31049765.0,-0.332171,-0.262360,-0.094641,-0.234995,"Papel Kraft, não revestidos"
4,Chile,42974952.0,43236511.0,25493817.0,0.006086,0.050882,-0.042626,0.039804,Carnes de aves
5,México,31617689.0,42872962.0,28663049.0,0.355980,0.487547,-0.088445,0.251365,Partes de motor
6,Filipinas,31245074.0,35617967.0,21509581.0,0.139955,0.054966,0.080561,0.055476,Carne suína
7,Arábia Saudita,31635023.0,29960918.0,21588422.0,-0.052919,-0.012019,-0.041398,0.033707,Carnes de aves
8,Emirados Árabes Unidos,22080050.0,29212236.0,21718821.0,0.323015,1.065554,-0.359487,0.076998,Carnes de aves
9,Países Baixos (Holanda),26816901.0,26295968.0,7670545.0,-0.019426,-0.226821,0.268237,-0.227500,Carnes de aves


TOP SETORES PRODUTOS ORDEM CORRIGIDA

In [25]:
from pathlib import Path
import duckdb

GOLD_DIR  = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\expimp").resolve()
MARTS_DIR = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\marts\jan").resolve()
MARTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------ helpers ------------------------------

def _month_labels_pt(ref_year: int, ref_month: int):
    meses_pt = ["Janeiro","Fevereiro","Março","Abril","Maio","Junho",
                "Julho","Agosto","Setembro","Outubro","Novembro","Dezembro"]
    nome = meses_pt[ref_month - 1]
    prev_label_val = f"{nome} {ref_year-1} (em US$ FOB)"
    curr_label_val = f"{nome} {ref_year} (em US$ FOB)"
    curr_label_kg  = f"{nome} {ref_year} (em quilogramas líquidos)"
    return nome, prev_label_val, curr_label_val, curr_label_kg

def _ident(s: str) -> str:
    return '"' + s.replace('"','""') + '"'

def _sql_in_list(values):
    """Return SQL IN list for UPPER(TRIM(sc_competitiva)) filtering."""
    if values is None:
        return None
    if isinstance(values, str):
        values = [values]
    cleaned = []
    for v in values:
        if v is None: 
            continue
        s = str(v).strip()
        if not s:
            continue
        s = s.replace("'", "''")  # escape single quotes
        cleaned.append(f"'{s.upper()}'")
    if not cleaned:
        return None
    return ", ".join(cleaned)

def _detect_ref_month(con, gold_glob):
    d_curr = con.execute(f"""
        SELECT MAX(MAKE_DATE(CAST(year AS INTEGER), CAST(co_mes AS INTEGER), 1))
        FROM read_parquet('{gold_glob}', union_by_name=true)
    """).fetchone()[0]
    if d_curr is None:
        raise RuntimeError("No data found in Gold to determine reference month.")
    return d_curr.year, d_curr.month

In [26]:
def build_mart_products_by_sc_competitiva_report(
    sc_competitiva,                       # str or list[str] (case-insensitive)
    ref_year: int | None = None,
    ref_month: int | None = None,
    kind: str = "EXP",
    top_n: int | None = None,             # None => ALL products in the category
    out_name: str = "mart_exp_produtos_por_categoria.parquet",
):
    """
    One row per product in the selected sc_competitiva category (or categories).
    Ranking basis (for optional top_n): current month vl_fob within the category.
    """
    flow_kind = (kind or "EXP").upper()
    if flow_kind not in ("EXP","IMP"):
        raise ValueError("kind must be 'EXP' or 'IMP'")

    con = duckdb.connect()
    gold_glob = rf"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\comexstat_ncm_sc.parquet"

    if ref_year is None or ref_month is None:
        ref_year, ref_month = _detect_ref_month(con, gold_glob)

    _, prev_label_val, curr_label_val, curr_label_kg = _month_labels_pt(ref_year, ref_month)
    in_list = _sql_in_list(sc_competitiva)
    cat_filter = f"AND sc_competitiva_norm IN ({in_list})" if in_list else ""

    limit_clause = f"WHERE rn <= {int(top_n)}" if (isinstance(top_n, int) and top_n > 0) else ""

    final_sql = f"""
    WITH base AS (
      SELECT
        kind,
        CAST(year AS INTEGER)     AS yr,
        CAST(co_mes AS INTEGER)   AS mo,
        co_pais,
        COALESCE(CAST(nm_pais_dim AS VARCHAR)) AS pais_nome,
        produto,
        UPPER(TRIM(produto))      AS produto_norm,
        TRIM(produto)             AS produto_clean,
        sc_competitiva,
        UPPER(TRIM(sc_competitiva)) AS sc_competitiva_norm,
        CAST(vl_fob     AS DECIMAL(20,4)) AS vl_fob,
        CAST(kg_liquido AS DECIMAL(20,6)) AS kg_liquido,
        CAST(qt_estat   AS DECIMAL(20,6)) AS qt_estat,
        MAKE_DATE(CAST(year AS INTEGER), CAST(co_mes AS INTEGER), 1) AS d
      FROM read_parquet('{gold_glob}', union_by_name=true)
      WHERE (vl_fob IS NULL OR vl_fob >= 0)
        AND (kg_liquido IS NULL OR kg_liquido >= 0)
    ),
    month_scope AS (
      SELECT
        MAKE_DATE({int(ref_year)}, {int(ref_month)}, 1) AS d_curr,
        MAKE_DATE({int(ref_year)}, {int(ref_month)}, 1) - INTERVAL 1 YEAR AS d_prev
    ),

    -- Rank products within category by CURRENT MONTH value
    prod_rank_curr AS (
      SELECT
        b.produto_norm,
        MIN(b.produto_clean) AS produto_display,
        MIN(b.sc_competitiva) AS sc_competitiva,
        SUM(b.vl_fob)        AS vl_fob_curr_month
      FROM base b
      JOIN month_scope m ON b.d = m.d_curr
      WHERE b.kind = '{flow_kind}'
        AND b.produto_norm IS NOT NULL 
        AND b.produto_norm <> ''
        {cat_filter.replace('sc_competitiva_norm', 'b.sc_competitiva_norm')}
      GROUP BY 1
    ),
    ordered_prod AS (
      SELECT *,
             ROW_NUMBER() OVER (ORDER BY vl_fob_curr_month DESC, produto_norm) AS rn
      FROM prod_rank_curr
    ),
    prod_set AS (
      SELECT * FROM ordered_prod
      {limit_clause}
    ),

    -- Main destination per product (CURRENT MONTH)
    main_dest AS (
      SELECT
        b.produto_norm,
        b.pais_nome,
        SUM(b.vl_fob) AS vl_fob_dest_curr,
        ROW_NUMBER() OVER (
          PARTITION BY b.produto_norm
          ORDER BY SUM(b.vl_fob) DESC, b.pais_nome
        ) AS rnk
      FROM base b
      JOIN month_scope m ON b.d = m.d_curr
      JOIN prod_set t ON b.produto_norm = t.produto_norm
      WHERE b.kind = '{flow_kind}'
        {cat_filter.replace('sc_competitiva_norm', 'b.sc_competitiva_norm')}
      GROUP BY b.produto_norm, b.pais_nome
    ),
    main_dest_pick AS (
      SELECT produto_norm, pais_nome AS principal_destino
      FROM main_dest WHERE rnk = 1
    ),

    -- Current month metrics
    curr_m AS (
      SELECT
        b.produto_norm,
        SUM(b.vl_fob)          AS vl_fob_curr,
        SUM(b.kg_liquido)      AS kg_curr,
        SUM(b.qt_estat)        AS qt_curr
      FROM base b
      JOIN prod_set s ON b.produto_norm = s.produto_norm
      JOIN month_scope m ON b.d = m.d_curr
      WHERE b.kind = '{flow_kind}'
      GROUP BY 1
    ),
    -- Previous year same month metrics
    prev_m AS (
      SELECT
        b.produto_norm,
        SUM(b.vl_fob)          AS vl_fob_prev,
        SUM(b.kg_liquido)      AS kg_prev,
        SUM(b.qt_estat)        AS qt_prev
      FROM base b
      JOIN prod_set s ON b.produto_norm = s.produto_norm
      JOIN month_scope m ON b.d = m.d_prev
      WHERE b.kind = '{flow_kind}'
      GROUP BY 1
    )

    SELECT
      t.produto_display                                        AS {_ident('Produtos Valor FOB US$')},
      COALESCE(CAST(p.vl_fob_prev AS DECIMAL(20,2)), 0)        AS {_ident(prev_label_val)},
      COALESCE(CAST(c.vl_fob_curr AS DECIMAL(20,2)), 0)        AS {_ident(curr_label_val)},
      COALESCE(CAST(c.kg_curr     AS DECIMAL(20,3)), 0)        AS {_ident(curr_label_kg)},
      CASE WHEN p.vl_fob_prev IS NULL OR p.vl_fob_prev = 0 THEN NULL
           ELSE (c.vl_fob_curr - p.vl_fob_prev) / p.vl_fob_prev END
                                                              AS {_ident('Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %)')},
      CASE WHEN p.kg_prev IS NULL OR p.kg_prev = 0 THEN NULL
           ELSE (c.kg_curr - p.kg_prev) / p.kg_prev END
                                                              AS {_ident('Variação de Quantidade Exportada - Quilograma Líquido (em %)')},
      CASE WHEN (p.kg_prev IS NULL OR p.kg_prev = 0) THEN NULL
           WHEN (c.kg_curr IS NULL OR c.kg_curr = 0) THEN NULL
           ELSE ((c.vl_fob_curr / NULLIF(c.kg_curr, 0)) - (p.vl_fob_prev / NULLIF(p.kg_prev, 0)))
                / NULLIF((p.vl_fob_prev / NULLIF(p.kg_prev, 0)), 0) END
                                                              AS {_ident('Variação do Preço Médio')},
      CASE WHEN p.qt_prev IS NULL OR p.qt_prev = 0 THEN NULL
           ELSE (c.qt_curr - p.qt_prev) / p.qt_prev END
                                                              AS {_ident('Variação da Quantidade Estatística')},
      md.principal_destino                                    AS {_ident('Principal destino dos produtos mais exportados por SC')}
    FROM prod_set t
    LEFT JOIN main_dest_pick md ON md.produto_norm = t.produto_norm
    LEFT JOIN curr_m c          ON c.produto_norm  = t.produto_norm
    LEFT JOIN prev_m p          ON p.produto_norm  = t.produto_norm
    ORDER BY t.rn
    """

    out_path = MARTS_DIR / out_name
    con.execute(f"""
      COPY ({final_sql})
      TO '{out_path.as_posix()}'
      (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1)
    """)
    preview = con.execute(f"SELECT * FROM read_parquet('{out_path.as_posix()}') LIMIT 10").fetchdf()
    con.close()
    print(f"mart ✔ {out_path}")
    display(preview)


In [27]:
build_mart_products_by_sc_competitiva_report("Madeira e Móveis", ref_year=2026, ref_month=2, kind="EXP", out_name="mart_exp_produtos_por_categoria.parquet")  

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mart ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\marts\jan\mart_exp_produtos_por_categoria.parquet


,Produtos Valor FOB US$,Fevereiro 2025 (em US$ FOB),Fevereiro 2026 (em US$ FOB),Fevereiro 2026 (em quilogramas líquidos),Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %),Variação de Quantidade Exportada - Quilograma Líquido (em %),Variação do Preço Médio,Variação da Quantidade Estatística,Principal destino dos produtos mais exportados por SC
0,Madeira serrada,36075100.0,23768002.0,49235383.0,-0.341152,-0.332296,-0.013263,-0.258917,Estados Unidos
1,Madeira compensada,18622450.0,15518156.0,28599413.0,-0.166696,-0.154806,-0.014068,-0.162344,Alemanha
2,Outros móveis,19657805.0,13561292.0,6222956.0,-0.310132,-0.285969,-0.033841,-0.439149,Estados Unidos
3,Obras de carpintaria para construções,21213193.0,13339032.0,7787453.0,-0.371192,-0.208992,-0.205055,-0.209008,Estados Unidos
4,Madeira MDF,7250361.0,5111336.0,13858639.0,-0.295023,-0.364549,0.109411,-0.334125,México
5,Lenha e desperdícios de madeira,3841376.0,4943021.0,21765980.0,0.286784,-0.183034,0.575076,-0.170149,Itália
6,Madeira em forma,11801297.0,4323070.0,3328268.0,-0.633678,-0.518777,-0.238770,-0.513328,Estados Unidos
7,Folheados,1997051.0,2183783.0,8507313.0,0.093504,0.045887,0.045528,0.030560,Índia
8,Painéis de madeira aglomerada,4232471.0,2003833.0,5716017.0,-0.526557,-0.530020,0.007369,-0.524938,Peru
9,"Edredons, almofadas e artigos semelhantes",1395563.0,1166231.0,243355.0,-0.164329,-0.151204,-0.015464,-0.123322,Uruguai


In [28]:
build_mart_products_by_sc_competitiva_report("Madeira e Móveis",ref_year=2026, ref_month=2, kind="IMP", out_name="mart_imp_produtos_por_categoria.parquet")  

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mart ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\marts\jan\mart_imp_produtos_por_categoria.parquet


,Produtos Valor FOB US$,Fevereiro 2025 (em US$ FOB),Fevereiro 2026 (em US$ FOB),Fevereiro 2026 (em quilogramas líquidos),Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %),Variação de Quantidade Exportada - Quilograma Líquido (em %),Variação do Preço Médio,Variação da Quantidade Estatística,Principal destino dos produtos mais exportados por SC
0,Assentos,9161599.0,9855255.0,3913520.0,0.075713,0.193138,-0.098417,0.080051,China
1,"Edredons, almofadas e artigos semelhantes",7701373.0,5968099.0,1869878.0,-0.225060,-0.152126,-0.086021,-0.246816,China
2,Outros móveis,2704071.0,3237799.0,1666718.0,0.197379,0.777290,-0.326289,0.071867,China
3,Outros artigos de madeira,1742812.0,1175909.0,902289.0,-0.325281,-0.352551,0.042119,-0.352551,China
4,Utensílios de madeira para cozinha,779930.0,486729.0,279078.0,-0.375932,-0.201425,-0.218524,-0.201425,China
5,Madeira densificada,0.0,455597.0,112467.0,NaN,NaN,NaN,NaN,Itália
6,Obras de cestaria,422605.0,424320.0,105880.0,0.004058,0.253893,-0.199247,0.253893,China
7,Ornamentos de madeira,222629.0,293803.0,74207.0,0.319698,0.523883,-0.133990,0.523883,China
8,Madeira serrada,311677.0,259888.0,613902.0,-0.166162,0.475985,-0.435064,0.441137,Argentina
9,Madeira compensada,75743.0,164582.0,234879.0,1.172900,-0.128221,1.492491,-0.170543,China


TOP PRODUTOS CATEGORIAS DESTINO

In [29]:
def build_mart_destinations_by_sc_competitiva_report(
    sc_competitiva,                       # str or list[str]
    ref_year: int | None = None,
    ref_month: int | None = None,
    kind: str = "EXP",
    top_n: int | None = None,             # None => ALL destinations that received category products
    out_name: str = "mart_exp_destinos_por_categoria.parquet",
):
    """
    One row per destination that received products in the selected sc_competitiva category.
    Ranking basis (for optional top_n): current month vl_fob within the category.
    """
    flow_kind = (kind or "EXP").upper()
    if flow_kind not in ("EXP","IMP"):
        raise ValueError("kind must be 'EXP' or 'IMP'")

    con = duckdb.connect()
    gold_glob = rf"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\gold\comexstat_ncm_sc.parquet"

    if ref_year is None or ref_month is None:
        ref_year, ref_month = _detect_ref_month(con, gold_glob)

    _, prev_label_val, curr_label_val, curr_label_kg = _month_labels_pt(ref_year, ref_month)
    in_list = _sql_in_list(sc_competitiva)
    cat_filter = f"AND sc_competitiva_norm IN ({in_list})" if in_list else ""
    limit_clause = f"WHERE rn <= {int(top_n)}" if (isinstance(top_n, int) and top_n > 0) else ""

    final_sql = f"""
    WITH base AS (
      SELECT
        kind,
        CAST(year AS INTEGER)     AS yr,
        CAST(co_mes AS INTEGER)   AS mo,
        co_pais,
        COALESCE(CAST(nm_pais_dim AS VARCHAR)) AS pais_nome,
        produto,
        UPPER(TRIM(produto))      AS produto_norm,
        TRIM(produto)             AS produto_clean,
        sc_competitiva,
        UPPER(TRIM(sc_competitiva)) AS sc_competitiva_norm,
        CAST(vl_fob     AS DECIMAL(20,4)) AS vl_fob,
        CAST(kg_liquido AS DECIMAL(20,6)) AS kg_liquido,
        CAST(qt_estat   AS DECIMAL(20,6)) AS qt_estat,
        MAKE_DATE(CAST(year AS INTEGER), CAST(co_mes AS INTEGER), 1) AS d
      FROM read_parquet('{gold_glob}', union_by_name=true)
      WHERE (vl_fob IS NULL OR vl_fob >= 0)
        AND (kg_liquido IS NULL OR kg_liquido >= 0)
    ),
    month_scope AS (
      SELECT
        MAKE_DATE({int(ref_year)}, {int(ref_month)}, 1) AS d_curr,
        MAKE_DATE({int(ref_year)}, {int(ref_month)}, 1) - INTERVAL 1 YEAR AS d_prev
    ),

    -- Rank destinations within category by CURRENT MONTH value
    dest_rank_curr AS (
      SELECT
        b.co_pais,
        MIN(b.pais_nome) AS pais_nome,
        SUM(b.vl_fob)    AS vl_fob_curr_month
      FROM base b
      JOIN month_scope m ON b.d = m.d_curr
      WHERE b.kind = '{flow_kind}'
        {cat_filter.replace('sc_competitiva_norm', 'b.sc_competitiva_norm')}
      GROUP BY 1
    ),
    ordered_dest AS (
      SELECT *,
             ROW_NUMBER() OVER (ORDER BY vl_fob_curr_month DESC, co_pais) AS rn
      FROM dest_rank_curr
    ),
    dest_set AS (
      SELECT * FROM ordered_dest
      {limit_clause}
    ),

    -- Top product per destination within category (CURRENT MONTH)
    main_prod_curr AS (
      SELECT
        b.co_pais,
        b.produto_norm,
        MIN(b.produto_clean) AS produto,
        SUM(b.vl_fob)        AS vl_fob_prod_curr,
        ROW_NUMBER() OVER (
          PARTITION BY b.co_pais
          ORDER BY SUM(b.vl_fob) DESC, b.produto_norm
        ) AS rnk
      FROM base b
      JOIN month_scope m ON b.d = m.d_curr
      JOIN dest_set t ON b.co_pais = t.co_pais
      WHERE b.kind = '{flow_kind}'
        AND b.produto_norm IS NOT NULL 
        AND b.produto_norm <> ''
        {cat_filter.replace('sc_competitiva_norm', 'b.sc_competitiva_norm')}
      GROUP BY b.co_pais, b.produto_norm
    ),
    main_prod_pick AS (
      SELECT co_pais, produto AS top_produto
      FROM main_prod_curr
      WHERE rnk = 1
    ),

    -- Current month metrics (restricted to category)
    curr_m AS (
      SELECT
        b.co_pais,
        SUM(b.vl_fob)     AS vl_fob_curr,
        SUM(b.kg_liquido) AS kg_curr,
        SUM(b.qt_estat)   AS qt_curr
      FROM base b
      JOIN month_scope m ON b.d = m.d_curr
      JOIN dest_set s ON b.co_pais = s.co_pais
      WHERE b.kind = '{flow_kind}'
        {cat_filter.replace('sc_competitiva_norm', 'b.sc_competitiva_norm')}
      GROUP BY 1
    ),
    -- Previous year same month metrics (restricted to category)
    prev_m AS (
      SELECT
        b.co_pais,
        SUM(b.vl_fob)     AS vl_fob_prev,
        SUM(b.kg_liquido) AS kg_prev,
        SUM(b.qt_estat)   AS qt_prev
      FROM base b
      JOIN month_scope m ON b.d = m.d_prev
      JOIN dest_set s ON b.co_pais = s.co_pais
      WHERE b.kind = '{flow_kind}'
        {cat_filter.replace('sc_competitiva_norm', 'b.sc_competitiva_norm')}
      GROUP BY 1
    )

    SELECT
      t.pais_nome                                            AS {_ident('Destinos Valor FOB US$')},
      COALESCE(CAST(p.vl_fob_prev AS DECIMAL(20,2)), 0)      AS {_ident(prev_label_val)},
      COALESCE(CAST(c.vl_fob_curr AS DECIMAL(20,2)), 0)      AS {_ident(curr_label_val)},
      COALESCE(CAST(c.kg_curr     AS DECIMAL(20,3)), 0)      AS {_ident(curr_label_kg)},
      CASE WHEN p.vl_fob_prev IS NULL OR p.vl_fob_prev = 0 THEN NULL
           ELSE (c.vl_fob_curr - p.vl_fob_prev) / p.vl_fob_prev END
                                                            AS {_ident('Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %)')},
      CASE WHEN p.kg_prev IS NULL OR p.kg_prev = 0 THEN NULL
           ELSE (c.kg_curr - p.kg_prev) / p.kg_prev END
                                                            AS {_ident('Variação de Quantidade Exportada - Quilograma Líquido (em %)')},
      CASE WHEN (p.kg_prev IS NULL OR p.kg_prev = 0) THEN NULL
           WHEN (c.kg_curr IS NULL OR c.kg_curr = 0) THEN NULL
           ELSE ((c.vl_fob_curr / NULLIF(c.kg_curr, 0)) - (p.vl_fob_prev / NULLIF(p.kg_prev, 0)))
                / NULLIF((p.vl_fob_prev / NULLIF(p.kg_prev, 0)), 0) END
                                                            AS {_ident('Variação do Preço Médio')},
      CASE WHEN p.qt_prev IS NULL OR p.qt_prev = 0 THEN NULL
           ELSE (c.qt_curr - p.qt_prev) / p.qt_prev END
                                                            AS {_ident('Variação da Quantidade Estatística')},
      mp.top_produto                                         AS {_ident('Produto mais exportado para o destino')}
    FROM dest_set t
    LEFT JOIN main_prod_pick mp ON mp.co_pais = t.co_pais
    LEFT JOIN curr_m c          ON c.co_pais  = t.co_pais
    LEFT JOIN prev_m p          ON p.co_pais  = t.co_pais
    ORDER BY t.rn
    """

    out_path = MARTS_DIR / out_name
    con.execute(f"""
      COPY ({final_sql})
      TO '{out_path.as_posix()}'
      (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1)
    """)
    preview = con.execute(f"SELECT * FROM read_parquet('{out_path.as_posix()}') LIMIT 10").fetchdf()
    con.close()
    print(f"mart ✔ {out_path}")
    display(preview)


In [30]:
build_mart_destinations_by_sc_competitiva_report("Madeira e Móveis", ref_year=2026, ref_month=2, kind="EXP", out_name="mart_exp_destinos_por_categoria.parquet")  

mart ✔ C:\Users\matheus.manzke\Projetos\dados-comex\Dados\marts\jan\mart_exp_destinos_por_categoria.parquet


,Destinos Valor FOB US$,Fevereiro 2025 (em US$ FOB),Fevereiro 2026 (em US$ FOB),Fevereiro 2026 (em quilogramas líquidos),Variação do valor(US$) em relação ao mesmo mês do ano anterior(em %),Variação de Quantidade Exportada - Quilograma Líquido (em %),Variação do Preço Médio,Variação da Quantidade Estatística,Produto mais exportado para o destino
0,Estados Unidos,58212931.0,26173602.0,23826267.0,-0.550382,-0.578069,0.065620,-0.285662,Obras de carpintaria para construções
1,México,5791862.0,8089096.0,17805702.0,0.396631,0.624354,-0.140193,1.938370,Madeira serrada
2,Itália,3870152.0,6122940.0,22302031.0,0.582093,0.242485,0.273329,0.229530,Lenha e desperdícios de madeira
3,Reino Unido,4642386.0,4116260.0,5031776.0,-0.113331,-0.022499,-0.092922,0.208442,Outros móveis
4,Alemanha,1553286.0,3907252.0,6621164.0,1.515475,2.510576,-0.283458,-0.315196,Madeira compensada
5,Arábia Saudita,6607929.0,2871023.0,6045233.0,-0.565518,-0.577443,0.028221,-0.539235,Madeira serrada
6,Espanha,2863146.0,2844045.0,4714979.0,-0.006671,0.502006,-0.338665,-0.407255,Madeira serrada
7,França,2764657.0,2475123.0,1555605.0,-0.104727,-0.218646,0.145797,0.189480,Outros móveis
8,Uruguai,2169508.0,2103533.0,1220017.0,-0.030410,0.123251,-0.136800,-0.057665,Outros móveis
9,Vietnã,2734069.0,1833735.0,4638867.0,-0.329302,-0.325567,-0.005537,-0.339616,Madeira serrada


# Automating reports

In [33]:
from pathlib import Path
from datetime import datetime
import pandas as pd

MARTS_DIR   = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\marts\jan").resolve()
REPORTS_DIR = Path(r"C:\Users\matheus.manzke\Projetos\dados-comex\Dados\reports").resolve()
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------- helpers ----------

def _is_var_col(name: str) -> bool:
    n = str(name).lower()
    return n.startswith("yoy_") or ("variação" in n) or ("variacao" in n)

def _is_numbery_column_name(col: str) -> bool:
    """Heuristic: columns that should be numeric even if read as object/Decimal."""
    c = str(col).strip().lower()
    # dynamic headers created in the reports
    if c.endswith("(em us$ fob)") or c.endswith("(em quilogramas líquidos)"):
        return True
    # common fixed metric columns
    number_cols = {
        "vl_fob_sum_last12m","vl_fob_12m","vl_fob_curr","vl_fob_prev","vl_fob",
        "kg_liquido","kg_curr","kg_prev",
        "qt_estat","qt_curr","qt_prev",
        "main_dest_vl_fob_last12m","main_prod_vl_fob_last12m",
        "rank_12m",  # optional: include if you want coloring on ranks
    }
    return c in number_cols

def _load_parquet(p: Path) -> pd.DataFrame:
    df = pd.read_parquet(p)

    # Dates
    if "ref_month" in df.columns:
        df["ref_month"] = pd.to_datetime(df["ref_month"], errors="coerce")

    # Explicit: variation/YoY columns -> float
    for c in df.columns:
        if _is_var_col(c):
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Heuristic: coerce numeric-like columns (handles Decimal/object/string)
    for c in df.columns:
        if _is_numbery_column_name(c):
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Fallback: if still nothing is numeric (all object/strings), try a safe sniff:
    if not any(pd.api.types.is_numeric_dtype(df[d]) for d in df.columns):
        for c in df.columns:
            if df[c].dtype == "object":
                probe = pd.to_numeric(df[c], errors="coerce")
                # adopt if at least 80% of non-nulls convert
                non_null = probe.notna().sum()
                total    = (~df[c].isna()).sum()
                if total > 0 and (non_null / total) >= 0.8:
                    df[c] = probe

    return df

def _autowidth(ws, df, wb, start_col=0, extra_pad=2, max_width=35):
    # Slight cap on width so headers actually wrap
    for i, col in enumerate(df.columns):
        max_len = max(len(str(col)), *(len(str(x)) for x in df[col].head(500)))
        ws.set_column(start_col + i, start_col + i, min(max_len + extra_pad, max_width))

def _apply_content_base(ws, df, wb, start_col=0):
    # Base content font for the whole table (overridden later per-type)
    fmt_content = wb.add_format({"font_name": "Aptos Narrow", "font_size": 11})
    ws.set_column(start_col, start_col + len(df.columns) - 1, None, fmt_content)

def _apply_number_formats(ws, df, wb, header_row, start_col=0):
    """
    Make all numeric columns integers (#,##0), except variação/yoy columns (percent).
    Dates keep yyyy-mm. Uses Aptos Narrow 11 everywhere.
    """
    import pandas as pd

    common = {"font_name": "Aptos Narrow", "font_size": 11}
    fmt_int  = wb.add_format({**common, "num_format": "#,##0"})   # integers
    fmt_pct  = wb.add_format({**common, "num_format": "0.0%"})    # percents
    fmt_date = wb.add_format({**common, "num_format": "yyyy-mm"})

    # detect columns
    pct_cols  = [c for c in df.columns if _is_var_col(c)]  # variação/yoy
    date_cols = [c for c in df.columns if c == "ref_month"]

    # numeric columns by dtype
    numeric_cols = list(df.select_dtypes(include="number").columns)

    # integers for numeric columns EXCEPT percent + date index columns
    int_cols = [c for c in numeric_cols if c not in pct_cols]

    # apply formats
    idx = {c: i for i, c in enumerate(df.columns)}

    for c in int_cols:
        j = idx.get(c, None)
        if j is not None:
            ws.set_column(start_col + j, start_col + j, None, fmt_int)

    for c in pct_cols:
        j = idx.get(c, None)
        if j is not None:
            ws.set_column(start_col + j, start_col + j, None, fmt_pct)

    for c in date_cols:
        j = idx.get(c, None)
        if j is not None:
            ws.set_column(start_col + j, start_col + j, 12, fmt_date)


def _style_headers(ws, df, wb, header_row, start_col=0):
    # Two header styles; wrap text; correct fonts/sizes
    hdr_main = wb.add_format({
        "font_name": "Aptos Narrow", "font_size": 12, "bold": True,
        "font_color": "white", "bg_color": "#0E2841", "border": 1,
        "text_wrap": True, "align": "center", "valign": "vcenter"
    })
    hdr_var  = wb.add_format({
        "font_name": "Aptos Narrow", "font_size": 12, "bold": True,
        "font_color": "white", "bg_color": "#215C98", "border": 1,
        "text_wrap": True, "align": "center", "valign": "vcenter"
    })
    # Give the header row some height to breathe for wrapping
    ws.set_row(header_row, 36)
    # Write each header with the right color
    for j, name in enumerate(df.columns):
        fmt = hdr_var if _is_var_col(name) else hdr_main
        ws.write(header_row, start_col + j, name, fmt)

def _apply_conditional_formats(ws, df, wb, data_first_row, data_last_row, start_col=0):
    """
    Apply conditional colors ONLY to 'Variação' columns (yoy_* or names containing 'Variação').
    Green fill+font for > 0, red fill+font for < 0. No zebra banding here.
    """
    if data_last_row < data_first_row:
        return

    # target only variation columns
    var_cols = [c for c in df.columns if _is_var_col(c)]
    if not var_cols:
        return

    fmt_green = wb.add_format({
        "font_name": "Aptos Narrow", "font_size": 11,
        "bg_color": "#C6EFCE", "font_color": "#006100"
    })
    fmt_red = wb.add_format({
        "font_name": "Aptos Narrow", "font_size": 11,
        "bg_color": "#FFC7CE", "font_color": "#9C0006"
    })

    for col in var_cols:
        j = start_col + df.columns.get_loc(col)
        # > 0 => green
        ws.conditional_format(data_first_row, j, data_last_row, j, {
            "type": "cell", "criteria": ">", "value": 0, "format": fmt_green
        })
        # < 0 => red
        ws.conditional_format(data_first_row, j, data_last_row, j, {
            "type": "cell", "criteria": "<", "value": 0, "format": fmt_red
        })

def _write_title(ws, title_text, wb, n_cols, start_col=0, row=0):
    title_fmt = wb.add_format({"bold": True, "font_size": 22, "font_name": "Aptos Narrow",
                               "font_color": "#0E2841", "valign": "vcenter"})
    ws.merge_range(row, start_col, row, start_col + n_cols - 1, title_text, title_fmt)

# ---------- main exporter ----------

def create_excel_report(
    marts_dir: Path = MARTS_DIR,
    out_path: Path = REPORTS_DIR / "comex_datamarts_2026_fev.xlsx",
    include_charts: bool = False  # charts disabled by default
):
    candidates = [

        ("mart_exp_top_products_report.parquet",     "EXP Top 20 produtos"),
        ("mart_exp_top_destinations_report.parquet", "EXP Top 20 destinos"),
        ("mart_exp_produtos_por_categoria.parquet",  "EXP Produtos por Categoria"),
        ("mart_exp_destinos_por_categoria.parquet",  "EXP Destinos por Categoria"),
        ("mart_imp_top_products_report.parquet",     "IMP Top 20 produtos"),
        ("mart_imp_top_destinations_report.parquet",  "IMP Top 20 origens"),
    ]
    sheets = [(marts_dir / f, name) for f, name in candidates if (marts_dir / f).exists()]
    if not sheets:
        raise FileNotFoundError(f"No known mart files found in {marts_dir}")

    dfs = [(name, _load_parquet(p)) for p, name in sheets]

    ref_month = None
    for _, df in dfs:
        if "ref_month" in df.columns and not df["ref_month"].isna().all():
            ref_month = pd.to_datetime(df["ref_month"].max()).date()
            break

    with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
        wb = writer.book

        # Cover sheet (keep simple but with Aptos font)
        cover = pd.DataFrame({
            "Item": ["Generated at", "Reference month", "Marts included"],
            "Value": [datetime.now().strftime("%Y-%m-%d %H:%M"), str(ref_month) if ref_month else "—", ", ".join([n for n,_ in dfs])]
        })
        cover.to_excel(writer, sheet_name="Cover", index=False)
        ws_cover = writer.sheets["Cover"]
        _autowidth(ws_cover, cover, wb)
        # apply a simple content base font on cover
        cover_fmt = wb.add_format({"font_name": "Aptos Narrow", "font_size": 11})
        ws_cover.set_column(0, cover.shape[1]-1, None, cover_fmt)
        ws_cover.freeze_panes(1, 0)
        ws_cover.autofilter(0, 0, cover.shape[0], cover.shape[1]-1)

        # Data sheets
        for sheet_name, df in dfs:
            df_out = df.copy()

            # Optional: reorder some common keys first
            preferred_first = [c for c in ["rank_12m","produto","ncm8","co_ncm","no_pais","co_pais","main_produto","main_no_pais","main_co_pais","vl_fob_sum_last12m"] if c in df_out.columns]
            other_cols = [c for c in df_out.columns if c not in preferred_first]
            df_out = df_out[preferred_first + other_cols] if preferred_first else df_out

            # Title at row 0; table header at row 1
            startrow = 1
            df_out.to_excel(writer, sheet_name=sheet_name, index=False, startrow=startrow)
            ws = writer.sheets[sheet_name]

            # Title
            _write_title(ws, sheet_name, wb, n_cols=df_out.shape[1], start_col=0, row=0)

            # Base content font (Aptos Narrow 11) over full table area
            _apply_content_base(ws, df_out, wb, start_col=0)

            # Header styling (wrapped, Aptos Narrow 12, colors)
            _style_headers(ws, df_out, wb, header_row=startrow, start_col=0)

            # Widths + number/percent/date formats
            _autowidth(ws, df_out, wb, start_col=0, max_width=35)
            _apply_number_formats(ws, df_out, wb, header_row=startrow, start_col=0)

            # Conditional formats on data rows
            data_first_row = startrow + 1
            data_last_row  = startrow + len(df_out)
            _apply_conditional_formats(ws, df_out, wb, data_first_row, data_last_row, start_col=0)

            # Freeze panes so both title and header are visible when scrolling
            ws.freeze_panes(startrow + 1, 0)

            # Autofilter across the table
            ws.autofilter(startrow, 0, startrow + df_out.shape[0], df_out.shape[1]-1)

    print(f"Excel report ✔  {out_path}")

# Usage:
# create_excel_report()


In [34]:
create_excel_report()

Excel report ✔  C:\Users\matheus.manzke\Projetos\dados-comex\Dados\reports\comex_datamarts_2026_fev.xlsx
